In [74]:
from pprint import pprint
import pathlib
import csv
import os
from utils import *
import csv
import contractions
import shutil
import emoji
from emot.emo_unicode import EMOTICONS_EMO
import re

In [75]:
# create preprocessed dir if not exists
pathlib.Path(preprocessed_data_path).mkdir(parents=True, exist_ok=True)

# copy raw reviews for preprocessing
shutil.copytree(raw_data_path, preprocessed_data_path, dirs_exist_ok=True)

def preprocess_data(process_fn):
    for name in os.listdir(preprocessed_data_path):
        path = os.path.join(preprocessed_data_path, name)

        with open(path, newline="") as f:
            reader = csv.DictReader(f)
            data = list(reader)

            for review in data:
                text = process_fn(review["content"])
                review["content"] = text

            write_path = os.path.join(preprocessed_data_path, name)
            file = open(write_path, "w")
            output_csv(data, file)

            file.close()

In [76]:
contractions_dict = {
    "i'mma": "i will"
}

def expand_contractions(content):
    expanded_words = []
    content_words = content.split(" ")
    for word in content_words:
        word_new = ""
        if word not in contractions_dict.keys():
            word_new = contractions.fix(word)
        else:
            word_new = contractions_dict[word]
        expanded_words.append(word_new)
    return " ".join(expanded_words)

preprocess_data(expand_contractions)

In [77]:
def remove_emojis(content):
    return emoji.replace_emoji(content, replace="")

preprocess_data(remove_emojis)

In [78]:
emoticons_dict_custom = EMOTICONS_EMO
emoticons_dict_custom["¯\\_(ツ)_/¯"] = "Shrug"

emoticon_regex = re.compile(
    "|".join(map(re.escape, emoticons_dict_custom.keys()))
)

def remove_emoticons(content):
    return re.sub(emoticon_regex, "", content)
preprocess_data(remove_emoticons)

In [79]:
def remove_stopwords(content):
    content_words = content.split(" ")
    filtered = [w for w in content_words if w not in STOPWORDS]

    return " ".join(filtered)

preprocess_data(remove_stopwords)